### PySpark: распределённые вычисления

In [ ]:
import time
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Локальный режим - использует все доступные ядра
spark = (SparkSession.builder
    .appName("Мой анализ")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

# Отключить лишние логи (для учебных целей)
spark.sparkContext.setLogLevel("ERROR")

print("Версия Spark:", spark.version)
print("Параллелизм по умолчанию (число ядер):", spark.sparkContext.defaultParallelism)

In [ ]:
spark.range(1).count()                                         # прогрев JVM, чтобы не искажать замеры ниже

t0 = time.perf_counter()
big = spark.range(5_000_000)                                   # трансформация: создаёт только план
big = big.withColumn("x", F.col("id") * 2).filter(F.col("x") % 3 == 0)
print(f"Построение плана:      {time.perf_counter() - t0:.3f}с  (ничего не посчитано)")

t0 = time.perf_counter()
n = big.count()                                                # ДЕЙСТВИЕ: запускает DAG
print(f"Действие count():      {time.perf_counter() - t0:.3f}с  → {n:,} строк")

print("Число партиций:", big.rdd.getNumPartitions())

## DataFrame API

In [ ]:
transactions = spark.read.csv('../data/transactions/transactions.csv')
transactions.count()

In [ ]:
transactions = spark.read.parquet('../data/transactions/transactions.parquet')
transactions.count()

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, DoubleType, TimestampType, BooleanType
)

N = 100_000_000

schema = StructType([
    StructField("id", LongType(), False),
    StructField("user", StringType(), True),
    StructField("region", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("active", BooleanType(), True),
    StructField("ts", TimestampType(), True),
])

df = (
    spark.range(0, N, numPartitions=16)                 
         .withColumnRenamed("id", "id")
         .withColumn("user", F.concat(F.lit("u_"), (F.col("id") % 1_000_000).cast("string")))
         .withColumn("region", F.element_at(
             F.array(*[F.lit(r) for r in ["NA","EU","APAC","LATAM","MEA"]]),
             (F.col("id") % 5 + 1).cast("int")))
         .withColumn("amount", (F.rand(seed=1) * 1000).cast("decimal(18, 4)"))
         .withColumn("active", (F.rand(seed=2) > 0.5))
         .withColumn("ts", F.date_add(F.lit("2026-01-01"), (F.col("id") % 730).cast("int")).cast("timestamp"))
         .select("id","user","region","amount","active","ts")
)

In [ ]:
df.show(5)

In [ ]:
(df.write
   .mode("overwrite")
   .option("compression", "gzip")
   .parquet('../data/transactions/big_data_transactions.parquet'))

print(df.count(), "rows")

### Создание DataFrame

In [ ]:
# Из dict
data = [
    ("Германия", 4.07, 2.1),
    ("Франция",  2.78, 2.5),
    ("Италия",   2.10, 3.0),
]
scheme = ["страна", "ввп", "инфляция"]
df_countries = spark.createDataFrame(data, schema=scheme)
df_countries.show()

In [ ]:
# Из CSV
df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/transactions/transactions.csv"))
df.printSchema()

### Базовые операции

In [ ]:
df = spark.read.parquet('../data/transactions/big_data_transactions.parquet')

In [ ]:
df.printSchema()

In [ ]:
# select - выбор столбцов
df.select("user", "amount")
df.select(F.col("amount"), F.col("region"))     # пока ничего не выполнено - это просто новый план

In [ ]:
# filter / where
df.filter(F.col("amount") > 1000)
df.where("`amount` > 1000")  

# withColumn - добавить/изменить столбец
df2 = df.withColumn("amount_rub", F.col("amount") * 100)

# drop
df2 = df2.drop("amount_rub")

# orderBy / sort
df.orderBy(F.col("amount").desc())
df.sort("amount", ascending=False)

# dropDuplicates
df.dropDuplicates(["user", "ts"])

#### Действия (запускают выполнение DAG)

In [ ]:
df.show(5)                    # напечатать первые 5 строк
df.show(5, truncate=False)    # не обрезать длинные строки

In [ ]:
print("Количество строк:", df.count())
print("Первая строка:   ", df.first())
print("Первые 3 строки: ", df.take(3))
df.describe("amount").show()    # описательная статистика

In [ ]:
# collect() переносит данные на driver
mini_df = df.limit(5).collect()
print(mini_df[:2])

### `withColumn` - преобразования

In [ ]:
df = spark.read.parquet('../data/transactions/big_data_transactions.parquet')

In [ ]:
df = (df
    # Арифметика
    .withColumn("amount_nds", F.col("amount") * 1.22)

    # Условие (аналог np.where)
    .withColumn("size",
        F.when(F.col("amount") > 1000, "крупная")
         .when(F.col("amount") > 100, "средняя")
         .otherwise("малая")
    )

    # Строки
    .withColumn("region", F.lower(F.trim(F.col("region"))))

    # Дата
    .withColumn("date", F.to_date(F.col("ts"), "yyyy-MM-dd"))
    .withColumn("year", F.year(F.col("ts")))
    .withColumn("month", F.month(F.col("ts")))

    # Заполнить пропуски
    .fillna({"amount": 0.0})
)

df.show(5)

### План запроса: `explain()`

In [ ]:
df.filter(F.col("amount") > 1000).select("id").explain()

### GroupBy, join и SQL-интерфейс

In [ ]:
df.printSchema()

In [ ]:
# Одна группировка
df.groupBy("month").agg(F.sum("amount").alias("result_region")).orderBy("user").show()

In [ ]:
# Несколько агрегаций
result_df = df.groupBy("month", "region").agg(
    F.sum("amount").alias("sum"),
    F.avg("amount").alias("avg"),
    F.count("*").alias("count"),
    F.max("amount").alias("max_amount"),
    F.countDistinct("user").alias("unique_users"),
)
result_df.orderBy(F.col("sum").desc()).show(10)

In [ ]:
# Оконные функции
from pyspark.sql.window import Window

window = Window.partitionBy("region").orderBy(F.col("amount").desc())
df_windowed = (df.withColumn("region_rank", F.rank().over(window))
    .withColumn("sum_over_window", F.sum("amount").over(window)))

df_windowed.select("region", "amount", "region_rank", "sum_over_window").show(5)

In [ ]:
df.groupBy("region").agg(F.sum("amount")).explain()

### Join

In [ ]:
clients = (
    spark.range(1, 100_001).withColumnRenamed("id", "client_id")
         .withColumn("client_name", F.concat(F.lit("client_"), F.col("client_id")))
         .withColumn("country",
             F.element_at(F.array(*[F.lit(c) for c in
                 ["AM","US","DE","FR","RU","GE","IT","ES"]]),
                 (F.col("client_id") % 8 + 1).cast("int")))
         .withColumn("signup_date",
             F.date_add(F.lit("2022-01-01"), (F.col("client_id") % 730).cast("int")))
)

products = (
    spark.range(1, 1_001).withColumnRenamed("id", "product_id")
         .withColumn("product_name", F.concat(F.lit("product_"), F.col("product_id")))
         .withColumn("category",
             F.element_at(F.array(*[F.lit(c) for c in
                 ["books","electronics","food","clothes","toys"]]),
                 (F.col("product_id") % 5 + 1).cast("int")))
         .withColumn("price", (F.rand(seed=10) * 500 + 5).cast("decimal(10,2)"))
)

transactions = (
    spark.range(0, 10_000_000, numPartitions=16)
         .withColumnRenamed("id", "txn_id")
         .withColumn("client_id",  (F.col("txn_id") % 100_000 + 1).cast("long"))
         .withColumn("product_id", (F.col("txn_id") % 1_000   + 1).cast("long"))
         .withColumn("qty",        (F.rand(seed=1) * 5 + 1).cast("int"))
         .withColumn("txn_date",
             F.date_add(F.lit("2024-01-01"), (F.col("txn_id") % 365).cast("int")))
)

In [ ]:
# LEFT JOIN
left_join_result = transactions.join(clients, on="client_id", how="left")
left_join_result.show(5)

In [ ]:
# Разные имена ключей: в справочнике ключ называется "id"
clients_2 = clients.withColumnRenamed("client_id", "id_client")

join_result = transactions.join(
    clients_2,
    transactions["client_id"] == clients_2["id_client"],
    how="inner",
)
# при таком синтаксисе оба ключа остаются в результате
join_result.show(3)

In [ ]:
# Broadcast join - для маленьких таблиц (избегает shuffle!)
from pyspark.sql.functions import broadcast

join_result = transactions.join(broadcast(products), on="product_id")
# Spark рассылает маленькую таблицу на все executors
join_result.explain()      # BroadcastHashJoin

In [ ]:
# Несколько join
multiple_join = (transactions
    .join(broadcast(clients),  "client_id",  "left")
    .join(broadcast(products), "product_id", "left")
)
multiple_join.show(5)

### SQL-интерфейс

In [ ]:
transactions.printSchema()

In [ ]:
# Зарегистрировать DataFrame как временную таблицу
transactions.createOrReplaceTempView("transactions")
clients.createOrReplaceTempView("clients")

# Обычный SQL!
sql_df = spark.sql("""
    SELECT
        c.country,
        t.txn_date,
        SUM(t.qty) AS sum_qty,
        AVG(t.qty) AS avg_qty,
        COUNT(*) AS cnt
    FROM transactions t
    LEFT JOIN clients c ON t.client_id = c.client_id
    WHERE t.qty > 0
    GROUP BY c.country, t.txn_date
    ORDER BY t.txn_date DESC
""")
sql_df.show(10)

In [ ]:
# Подзапросы, CTE, оконные функции - всё поддерживается
spark.sql("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', txn_date) AS month,
            SUM(qty) AS sum_qty
        FROM transactions
        GROUP BY 1
    )
    SELECT
        month,
        ROUND(sum_qty, 0) AS sum_qty,
        ROUND(LAG(sum_qty) OVER (ORDER BY month), 0) AS qty_last_month
    FROM monthly
    ORDER BY month
""").show()

### SQL и DataFrame API - `explain()`

In [ ]:
sql_df = spark.sql("SELECT txn_date, SUM(qty) as sum_qty FROM transactions GROUP BY txn_date")
api_df = transactions.groupBy("txn_date").agg(F.sum("qty").alias('sum_qty'))

print("=== SQL ===");          sql_df.explain()
print("=== DataFrame API ==="); api_df.explain()